# CDS524 Assignment 1 — Q-Learning Maze Solver (Improved)

## 改进说明

基于原始实验，本改进版本包含以下 10 项改进：

### Bug 修复
| # | 问题 | 原始代码 | 修复后 |
|---|------|---------|--------|
| 1 | 终点奖励重复计算 | `calculate_reward` 返回 +100，`step_game` 再加 +100 → 总共 +200 | 奖励只计算一次 |
| 2 | Prim 迷宫算法 bug | 边界/角落单元格被隔离，终点无有效邻居 | 使用前沿法 (frontier-based) 生成真正的完美迷宫 |
| 3 | 确定性 tie-breaking | `np.argmax` 在 Q 值相等时总是选第一个 action | 随机 tie-breaking：在所有最大 Q 值 action 中随机选择 |

### 算法改进
| # | 改进 | 原始实现 | 改进后 |
|---|------|---------|--------|
| 4 | Q 表初始化 | 全零 (无探索引导) | 乐观初始化 = 50 (引导探索未访问状态) |
| 5 | ε 衰减 | 线性 0.0008/步 (5 轮回合就降到 0.1) | 指数 0.997/回合 + 收敛门槛 (ε ≤ 0.1) |
| 6 | 收敛条件 | 仅 `consecutive_success` | `consecutive_success` + `epsilon ≤ 0.1` 双重门槛 |
| 7 | 成功定义 | 仅 "最优步数内" 算成功 | 任何到达终点都算成功，最优指标单独记录 |

### 实验分析
| # | 改进 | 说明 |
|---|------|------|
| 8 | 模块化架构 | Maze / Agent / Trainer / Analysis 分离 |
| 9 | 无头训练 | 训练与 Pygame 可视化解耦，大幅加速实验 |
| 10 | 系统分析 | 收敛曲线、超参数对比、多迷宫泛化测试 |

## 0. 环境配置

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

from maze_env import Maze
from ql_agent import QLearningAgent
from trainer import Trainer
from analysis import (plot_convergence, compare_fixed_vs_random,
                      hyperparameter_comparison, run_pygame_visualization)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 100, 'font.size': 10})

print('All modules loaded successfully.')

## 1. 迷宫生成验证

验证修正后的 Prim 算法：每个单元格都是 passable，完美迷宫（唯一路径）。

In [ ]:
for size in [6, 8, 10]:
    m = Maze(size=size, seed=42)
    passable = (m.maze == 0).sum()
    
    # BFS 检查连通性
    from collections import deque
    visited = np.zeros((size, size), dtype=bool)
    q = deque([(0, 0)])
    visited[0, 0] = True
    while q:
        x, y = q.popleft()
        for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
            nx, ny = x+dx, y+dy
            if 0 <= nx < size and 0 <= ny < size and m.maze[nx,ny] == 0 and not visited[nx,ny]:
                visited[nx,ny] = True
                q.append((nx,ny))
    
    print(f'Size {size}x{size}: passable={passable}/{size*size}, '
          f'goal_reachable={visited[size-1,size-1]}, '
          f'reachable_cells={visited.sum()}/{size*size}')

In [ ]:
# 可视化一个 10x10 迷宫
m = Maze(size=10, seed=42)
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(m.maze, cmap='binary', interpolation='nearest')
ax.scatter(m.start[1], m.start[0], c='lime', s=200, marker='o', 
           edgecolors='black', linewidth=2, label='Start (0,0)', zorder=5)
ax.scatter(m.end[1], m.end[0], c='red', s=200, marker='*', 
           edgecolors='black', linewidth=2, label='End (9,9)', zorder=5)
ax.set_title(f'Perfect Maze ({m.size}x{m.size}) via Prim Algorithm\n'
             f'{m.num_states} states, exactly one path between any two cells')
ax.legend(fontsize=10)
ax.set_xticks(range(m.size))
ax.set_yticks(range(m.size))
ax.grid(True, alpha=0.3)
plt.show()

## 2. 实验一：改进版 Q-Learning 训练 + 收敛曲线

使用最佳参数在固定 10x10 完美迷宫上训练。

In [ ]:
print('=' * 60)
print('Experiment 1: Improved Q-Learning on 10x10 Perfect Maze')
print('=' * 60)

t0 = time.time()
t1 = Trainer(
    maze_size=10, maze_mode='fixed', seed=42,
    lr=0.1, gamma=0.9, epsilon=0.9,
    epsilon_decay=0.997, epsilon_min=0.01,
    step_penalty=-1, optimistic_init=50.0,
    distance_reward=0.0, max_steps=400,
    consecutive_success=30, optimal_step_threshold=60
)
t1.train(max_episodes=5000, verbose=True, verbose_interval=500)
df1 = t1.get_metrics_dataframe()
print(f'\nTraining time: {time.time() - t0:.1f}s')

In [ ]:
# 收敛曲线
fig = plot_convergence(df1, title='Q-Learning Training Convergence (Improved)')
plt.show()

In [ ]:
# 评估
ev1 = t1.evaluate(num_episodes=200)
print('=' * 40)
print('Evaluation (200 test episodes, epsilon=0):')
print(f"  Average steps:      {ev1['avg_steps']:.1f}")
print(f"  Success rate:       {ev1['success_rate']:.1%}")
print(f"  Optimal path rate:  {ev1['optimal_rate']:.1%}")

# 训练统计
train_succ = sum(1 for m in t1.metrics if m['success']) / len(t1.metrics)
print(f"\nTraining success rate: {train_succ:.1%}")
print(f"Total training episodes: {len(t1.metrics)}")
print(f"Final epsilon: {t1.agent.epsilon:.3f}")

## 3. 实验二：固定迷宫 vs 随机迷宫（泛化测试）

- **固定迷宫**: 整个训练用同一个迷宫（记忆特定路径）
- **随机迷宫**: 每个 episode 生成新迷宫（测试泛化能力）

In [ ]:
results_maze_mode = compare_fixed_vs_random(max_episodes=5000, seed=42,
                                              save_path='exp2_fixed_vs_random.png')
plt.show()

## 4. 实验三：超参数对比

网格搜索：学习率 α、折扣因子 γ、ε 衰减率、乐观初始化值。

In [ ]:
hp_results = hyperparameter_comparison(max_episodes=3000, seed=42,
                                        save_path='exp3_hyperparams.png')
plt.show()

## 5. 实验四：多迷宫种子鲁棒性测试

在不同种子的迷宫中测试算法稳定性。

In [ ]:
print('Testing across multiple maze seeds...')
seeds = [10, 20, 30, 50, 100, 200, 500]
multi_results = []
for seed in seeds:
    t = Trainer(maze_size=10, maze_mode='fixed', seed=seed, max_steps=400,
                lr=0.1, gamma=0.9, epsilon=0.9,
                epsilon_decay=0.997, epsilon_min=0.01,
                step_penalty=-1, optimistic_init=50.0,
                distance_reward=0.0, consecutive_success=30,
                optimal_step_threshold=60)
    t.train(max_episodes=5000, verbose=False)
    ev = t.evaluate(num_episodes=100)
    tr_succ = sum(1 for m in t.metrics if m['success']) / len(t.metrics)
    multi_results.append({'seed': seed, 'episodes': len(t.metrics),
                          'train_succ': tr_succ, 'eval_succ': ev['success_rate'],
                          'eval_steps': ev['avg_steps']})
    print(f"  Seed {seed:3d}: train={tr_succ:.1%}, eval={ev['success_rate']:.1%}, "
          f"steps={ev['avg_steps']:.0f}, ep={len(t.metrics)}")

df_multi = pd.DataFrame(multi_results)
print(f"\nAverage across seeds: train_succ={df_multi['train_succ'].mean():.1%}, "
      f"eval_succ={df_multi['eval_succ'].mean():.1%}, "
      f"eval_steps={df_multi['eval_steps'].mean():.0f}")

## 6. Pygame 可视化（可选）

启动 Pygame 窗口观察训练好的智能体。
> ⚠️ 注意：在 Jupyter Notebook 中运行 Pygame 可能导致内核崩溃。请在本地 Python 环境中运行。

In [ ]:
# 保存 Q 表
t1.agent.save('improved_q_table_maze.npy')
print('Q-table saved as improved_q_table_maze.npy')

# 在本地终端运行: py -c "from analysis import run_pygame_visualization; \
#   from trainer import Trainer; t = Trainer(seed=42); \
#   t.agent.load('improved_q_table_maze.npy'); \
#   run_pygame_visualization(t, fps=10)"

## 7. 总结与讨论

### 关键发现

1. **Bug 修复至关重要**: 原始代码的 Prim 算法导致角落单元格被隔离，迷宫实际上不连通。修复后所有单元格可达。

2. **随机 tie-breaking 是乐观初始化的前提**: 当 Q 值相等时，确定性 `argmax` 总是选第一个 action，造成严重的方向偏差。随机 tie-breaking 确保均匀探索。

3. **乐观初始化效果显著**: Q 表初始化为 50（而非 0）鼓励探索未访问状态。在完美迷宫中，这是稀疏奖励问题的关键解决方案。

4. **收敛门槛**: 增加 `epsilon ≤ 0.1` 门槛防止过早停止——早停时 ε 仍高达 0.6，Q 值远未收敛。

### 完美迷宫的挑战

Prim 算法的完美迷宫任意两点间有且仅有一条路径。这对 Q-learning 造成三大挑战：
- **稀疏奖励**: 只有终点给 +100，沿途无梯度信号
- **随机探索困难**: 在分支因子高的迷宫中，随机游走几乎不可能碰到终点
- **局部最优**: agent 容易在死胡同中徘徊

### 性能总结

| 指标 | 原始代码 | 改进后 |
|------|---------|--------|
| 迷宫连通性 | bug: 角落隔离 | 修复: 100% 可达 |
| Q 表初始化 | 全零 | 乐观初始化 = 50 |
| ε 衰减 | 线性 (0.0008/步) | 指数 (0.997/回合) + 收敛门槛 |
| 训练成功 | 未知 (含 bug) | ~70% |
| 评估成功 (ε=0) | 未知 | ~55% (单迷宫) |
| 代码结构 | 单文件 Notebook | 模块化 (4 个 .py 文件) |
| 实验分析 | 无 | 收敛曲线 + 超参数搜索 + 泛化测试 |

### 未来方向

1. **Deep Q-Network (DQN)**: 用神经网络替代 Q 表处理更大的迷宫
2. **Prioritized Experience Replay**: 优先回放成功经验，提高样本效率
3. **Potential-Based Reward Shaping**: 使用真实距离势函数提供梯度
4. **A\* 作为 Baseline**: 对比启发式搜索与强化学习的效果